# 01 Data Audit

Local notebook for raw data auditing and preprocessing output checks.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'vn30f1m.csv'
print('PROJECT_ROOT:', PROJECT_ROOT)
print('RAW_PATH exists:', RAW_PATH.exists())

PROJECT_ROOT: C:\Users\votranhuonggiang\OneDrive\ドキュメント\Python\MiQuant\Training\MomentumTransformer\vn30f1m_momentum_transformer
RAW_PATH exists: True


In [2]:
df = pd.read_csv(RAW_PATH)
print(df.head())
print(df.columns.tolist())
print('rows:', len(df))

             timestamp   open   high    low  close  volume
0  2017-11-06 09:00:00  841.0  841.7  840.6  840.6   253.0
1  2017-11-06 09:01:00  841.0  841.0  840.7  841.0    43.0
2  2017-11-06 09:02:00  841.1  841.5  841.1  841.5    37.0
3  2017-11-06 09:03:00  841.6  842.0  841.6  842.0    56.0
4  2017-11-06 09:04:00  841.8  841.9  841.6  841.9    49.0
['timestamp', 'open', 'high', 'low', 'close', 'volume']
rows: 512906


In [3]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp')
print('min_ts:', df['timestamp'].min())
print('max_ts:', df['timestamp'].max())
print('duplicate_ts:', int(df['timestamp'].duplicated().sum()))

min_ts: 2017-11-06 09:00:00
max_ts: 2026-05-11 14:45:00
duplicate_ts: 0


In [4]:
bad_ohlc = ((df[['open','high','low','close']] <= 0).any(axis=1)).sum()
bad_hi = (df['high'] < df[['open','close']].max(axis=1)).sum()
bad_lo = (df['low'] > df[['open','close']].min(axis=1)).sum()
print('bad_ohlc<=0:', int(bad_ohlc))
print('bad_high_rule:', int(bad_hi))
print('bad_low_rule:', int(bad_lo))

bad_ohlc<=0: 0
bad_high_rule: 0
bad_low_rule: 0


In [5]:
import subprocess, sys
cmd = [sys.executable, str(PROJECT_ROOT / 'src' / 'data_preprocessing.py'), '--config', str(PROJECT_ROOT / 'configs' / 'default.yaml')]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)
print('Done preprocessing.')

Running: c:\Users\votranhuonggiang\AppData\Local\Programs\Python\Python311\python.exe C:\Users\votranhuonggiang\OneDrive\ドキュメント\Python\MiQuant\Training\MomentumTransformer\vn30f1m_momentum_transformer\src\data_preprocessing.py --config C:\Users\votranhuonggiang\OneDrive\ドキュメント\Python\MiQuant\Training\MomentumTransformer\vn30f1m_momentum_transformer\configs\default.yaml
Done preprocessing.


In [6]:
report = PROJECT_ROOT / 'outputs' / 'data_quality_report.md'
print('report exists:', report.exists())
print(report.read_text(encoding='utf-8')[:2000])

report exists: True
# Data Quality Report

## Summary
- Rows: 512906
- Date range: 2017-11-06 09:00:00 -> 2026-05-11 14:45:00
- Trading days: 2119

## Integrity Checks
- Non-positive OHLC rows: 0
- High/Low inconsistency rows: 0
- Extreme price jump rows (|log ret| > 3%): 21
- Extreme volume outlier rows (IQR rule): 3112
- Missing timestamps vs inferred session grid: 1410
- Inferred session time points per day: 240

## Notes
- Input data was sorted by timestamp and duplicate timestamps removed.
- Session grid was inferred from recurring intraday timestamps across days.
